In [127]:
import pandas as pd
# Load datasets
haunted_places_df = pd.read_csv("../data/haunted_places.tsv", sep='\t')
ucr_by_state_df = pd.read_csv("../data/ucr_by_state.csv")

In [128]:
# 1. Data Cleaning and Preparation
# Convert relevant columns to numeric (removing commas)
ucr_by_state_df["violent_crime_total"] = ucr_by_state_df["violent_crime_total"].astype(str).str.replace(",", "").astype(float)
ucr_by_state_df["murder_manslaughter"] = ucr_by_state_df["murder_manslaughter"].astype(str).str.replace(",", "").astype(float)
ucr_by_state_df["property_crime_total"] = ucr_by_state_df["property_crime_total"].astype(str).str.replace(",", "").astype(float)

In [129]:
# 2. Compute the average crime statistics for each state across all years
# Compute the average crime statistics per state
ucr_avg_df = ucr_by_state_df.groupby("jurisdiction").agg({
    "violent_crime_total": "mean",
    "murder_manslaughter": "mean",
    "property_crime_total": "mean"
}).reset_index()

# Rename columns for clarity
ucr_avg_df.rename(columns={
    "jurisdiction": "State",
    "violent_crime_total": "avg_violent_crime",
    "murder_manslaughter": "avg_murder_manslaughter",
    "property_crime_total": "avg_property_crime"
}, inplace=True)

In [130]:
# 3. Find the correct column representing 'State'
for col in haunted_places_df.columns:
    if "state" in col.lower():
        haunted_places_df.rename(columns={col: "State"}, inplace=True)
        break

Now, both datasets have a “State” column, so we can merge them.

In [131]:
# Merge Haunted Places dataset with UCR crime data on 'State'
merged_df = haunted_places_df.merge(ucr_avg_df, on="State", how="left")

In [132]:
# Save the merged dataset as a new file
merged_df.to_csv("merged_haunted_places_1.csv", index=False, header=True)
# Display first few rows
print(merged_df.head())

      city        country                                        description  \
0      Ada  United States  Ada witch - Sometimes you can see a misty blue...   
1  Addison  United States  A little girl was killed suddenly while waitin...   
2   Adrian  United States  If you take Gorman Rd. west towards Sand Creek...   
3   Adrian  United States  In the 1970's, one room, room 211, in the old ...   
4   Albion  United States  Kappa Delta Sorority - The Kappa Delta Sororit...   

                   location     State state_abbrev  longitude   latitude  \
0              Ada Cemetery  Michigan           MI -85.504893  42.962106   
1           North Adams Rd.  Michigan           MI -84.381843  41.971425   
2             Ghost Trestle  Michigan           MI -84.035656  41.904538   
3  Siena Heights University  Michigan           MI -84.017565  41.905712   
4            Albion College  Michigan           MI -84.745177  42.244006   

   city_longitude  city_latitude  ...  Percent Deaths from Acu